# 01 Load and merge

Loads the three PanPred tables, checks that strain IDs match and aligns them so that row *i* is the same strain everywhere.

Outputs:
- `X`: gene presence/absence (model input)
- `labels`: R/S lab results for 12 antibiotics
- `year`: collection year (time split only)
- `clusters`: lineage clusters at every SNP cutoff (lineage split only)

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/PanPred/test_data")
ANTIBIOTICS = ["CTZ", "CTX", "AMP", "AMX", "AMC", "TZP", "CXM", "CET", "GEN", "TBM", "TMP", "CIP"]

## Load the three tables

In [5]:
metadata = pd.read_csv(DATA_DIR / "Metadata.csv", index_col="Isolate")
metadata["Year"] = metadata["Year"].astype("Int64")

genes = pd.read_csv(DATA_DIR / "AccessoryGene.csv", index_col=0).astype("uint8")
genes.index.name = "Isolate"

population = pd.read_csv(DATA_DIR / "PopulationStructure.csv", index_col=0)
population.index.name = "Isolate"

## Check strain IDs

In [7]:
def describe_ids(name, df):
    print(f"{name:<12} rows: {len(df):>5}   unique IDs: {df.index.nunique():>5}   "
          f"duplicates: {df.index.duplicated().sum()}")

describe_ids("Metadata", metadata)
describe_ids("Genes", genes)
describe_ids("Population", population)

Metadata     rows:  1936   unique IDs:  1936   duplicates: 0
Genes        rows:  2033   unique IDs:  2033   duplicates: 0
Population   rows:  1936   unique IDs:  1936   duplicates: 0


In [8]:
meta_ids, gene_ids, pop_ids = set(metadata.index), set(genes.index), set(population.index)
common_ids = sorted(meta_ids & gene_ids & pop_ids)

print("In all three tables:", len(common_ids))
print("In metadata but not genes:", len(meta_ids - gene_ids))
print("In metadata but not population:", len(meta_ids - pop_ids))
print("In genes but not metadata:", len(gene_ids - meta_ids))

In all three tables: 1936
In metadata but not genes: 0
In metadata but not population: 0
In genes but not metadata: 97


## Align the tables

All 1,936 strains with lab results have gene and lineage data. The 97 genomes without lab results have no lab results and are dropped.

In [10]:
metadata = metadata.loc[common_ids]
genes = genes.loc[common_ids]
population = population.loc[common_ids]

assert metadata.index.equals(genes.index), "metadata and genes are not aligned"
assert metadata.index.equals(population.index), "metadata and population are not aligned"
assert metadata.index.is_unique, "duplicate strain IDs"
print(f"Aligned: {len(common_ids)} strains in all three tables")

Aligned: 1936 strains in all three tables


## Separate model input from split information

The model sees only `X`. Year and clusters are used only to build the train/test splits.

In [12]:
X = genes
labels = metadata[ANTIBIOTICS]
year = metadata["Year"]
clusters = population

assert X.columns.intersection(metadata.columns).empty, "metadata columns found in X"
assert X.columns.intersection(population.columns).empty, "cluster columns found in X"
print("X (strains × genes):", X.shape)
print("labels (strains × antibiotics):", labels.shape)

X (strains × genes): (1936, 17198)
labels (strains × antibiotics): (1936, 12)


## Quality checks

In [14]:
assert X.isna().sum().sum() == 0, "missing values in X"
assert np.isin(X.to_numpy(), [0, 1]).all(), "X contains values other than 0 and 1"

print("Strains without a year:", year.isna().sum())

gene_counts = X.sum()
always_absent = (gene_counts == 0).sum()
always_present = (gene_counts == len(X)).sum()
print("Genes absent in all strains: ", always_absent)
print("Genes present in all strains:", always_present)

Strains without a year: 58
Genes absent in all strains:  252
Genes present in all strains: 13


In [15]:
is_constant = (gene_counts == 0) | (gene_counts == len(X))
X = X.loc[:, ~is_constant]

assert X.shape[1] == 17198 - 265, "unexpected number of genes after removal"
print("Removed constant genes:", is_constant.sum())
print("X after removal:", X.shape)

Removed constant genes: 265
X after removal: (1936, 16933)


## Summary

- 1,936 strains with lab results, all with gene and lineage data
- 97 genomes without lab results dropped
- X: 1,936 strains × 16,933 genes, binary, no missing values
- 265 constant genes removed (252 absent everywhere, 13 present everywhere)
- Strains without a year: 58
- Constant genes: 252 absent everywhere, 13 present everywhere